# Colab Run – TS2IMG LightCNN Experiments

Notebook này dùng để chạy thực nghiệm cho repo `ts2img-lightcnn` trên Google Colab.

Quy trình gồm:

1. Mount Google Drive.
2. Clone hoặc cập nhật repo từ GitHub.
3. Cài thư viện, gồm `aeon`.
4. Kiểm tra GPU.
5. Test loader 5 bộ dữ liệu: GunPoint, ECG200, Coffee, FordA, Wafer.
6. Test nhanh pipeline.
7. Xóa kết quả test.
8. Chạy thực nghiệm chính thức.
9. Tạo các bảng tổng hợp dùng đưa vào bài báo.

Khuyến nghị: vào `Runtime → Change runtime type → GPU` trước khi chạy.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone hoặc cập nhật repo từ GitHub

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/hoangtnedu/ts2img-lightcnn.git"
REPO_DIR = Path("/content/drive/MyDrive/ts2img-lightcnn")
BRANCH = "main"

if (REPO_DIR / ".git").exists():
    print(f"Repo already exists: {REPO_DIR}")
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print(f"Cloning repo to: {REPO_DIR}")
    %cd /content/drive/MyDrive
    !git clone {REPO_URL}
    %cd {REPO_DIR}

print("Current directory:")
!pwd
print("\nLatest commit:")
!git log --oneline -1

## 3. Cài đặt thư viện

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. Kiểm tra code loader đã có aeon

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

print("Check requirements.txt:")
!grep -n "aeon" requirements.txt || true

print("\nCheck src/data_ucr.py:")
!grep -n "load_classification" src/data_ucr.py || true
!grep -n "Loading dataset with aeon" src/data_ucr.py || true

## 5. Kiểm tra GPU và phiên bản thư viện

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import aeon

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print("\nWARNING: Colab hiện chưa nhận GPU. Vào Runtime → Change runtime type → GPU.")

## 6. Test loader cho 5 bộ dữ liệu

In [ ]:
from src.data_ucr import load_dataset

datasets = ["GunPoint", "ECG200", "Coffee", "FordA", "Wafer"]

for name in datasets:
    X_train, X_test, y_train, y_test, num_classes, encoder = load_dataset(name)
    print(
        f"{name:10s} | "
        f"X_train={X_train.shape} | "
        f"X_test={X_test.shape} | "
        f"classes={num_classes}"
    )

## 7. Xóa kết quả cũ trước khi test pipeline

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!rm -rf runs/*
!rm -f results/summary_results.csv

print("Old runs and summary_results.csv were removed.")

## 8. Test nhanh pipeline

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!python -m src.run_experiments \
  --datasets GunPoint,ECG200,Coffee,FordA,Wafer \
  --representations gaf,rp \
  --model_type light2dcnn \
  --seeds 42 \
  --epochs 2 \
  --batch_size 16 \
  --image_size 64

## 9. Kiểm tra kết quả test nhanh

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary_results.csv")

print("Shape:", df.shape)
print("\nDataset counts:")
print(df["dataset"].value_counts())

print("\nRepresentation counts:")
print(df["representation"].value_counts())

print("\nSeed counts:")
print(df["seed"].value_counts())

display(df.tail(15))

## 10. Xóa kết quả test nhanh trước khi chạy chính thức

Chạy cell này trước khi chạy thực nghiệm chính thức để file kết quả không bị lẫn với kết quả test.

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!rm -rf runs/*
!rm -f results/summary_results.csv

print("Test results were removed. Ready for full experiment.")

## 11. Chạy thực nghiệm chính thức

Cell này chạy:

`5 dataset × 3 seed × 5 phương pháp = 75 lần huấn luyện`

Các phương pháp gồm:

- 1D-CNN baseline
- GAF + Light 2D-CNN
- MTF + Light 2D-CNN
- RP + Light 2D-CNN
- STFT + Light 2D-CNN

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!python -m src.run_experiments \
  --datasets GunPoint,ECG200,Coffee,FordA,Wafer \
  --representations gaf,mtf,rp,stft \
  --model_type light2dcnn \
  --seeds 42,2024,2026 \
  --epochs 50 \
  --batch_size 32 \
  --image_size 64

## 12. Kiểm tra kết quả chính thức

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary_results.csv")

print("Shape:", df.shape)

print("\nDataset counts:")
print(df["dataset"].value_counts())

print("\nRepresentation counts:")
print(df["representation"].value_counts())

print("\nSeed counts:")
print(df["seed"].value_counts())

print("\nExpected:")
print("dataset: each 15")
print("representation: each 15")
print("seed: each 25")

display(df.tail(20))

## 13. Tạo bảng tổng hợp theo dataset và phương pháp

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary_results.csv")

df["method"] = df.apply(
    lambda r: "1D-CNN" if r["representation"] == "none"
    else r["representation"].upper() + " + Light 2D-CNN",
    axis=1
)

summary = df.groupby(["dataset", "method"]).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    params_mean=("params", "mean"),
    train_time_mean=("training_time_sec", "mean"),
    infer_time_mean=("inference_time_per_sample_sec", "mean")
).reset_index()

summary.to_csv("results/summary_by_dataset_method.csv", index=False)

display(summary)
print("Saved: results/summary_by_dataset_method.csv")

## 14. Tạo bảng Accuracy / Macro F1-score để đưa vào bài báo

In [ ]:
paper_table_source = summary.copy()

paper_table_source["result"] = paper_table_source.apply(
    lambda r: f"{r['accuracy_mean']:.4f} / {r['macro_f1_mean']:.4f}",
    axis=1
)

paper_table = paper_table_source.pivot(
    index="dataset",
    columns="method",
    values="result"
).reset_index()

preferred_columns = [
    "dataset",
    "1D-CNN",
    "GAF + Light 2D-CNN",
    "MTF + Light 2D-CNN",
    "RP + Light 2D-CNN",
    "STFT + Light 2D-CNN",
]

paper_table = paper_table[[c for c in preferred_columns if c in paper_table.columns]]

paper_table.to_csv("results/table_accuracy_f1_for_paper.csv", index=False)

display(paper_table)
print("Saved: results/table_accuracy_f1_for_paper.csv")

## 15. Tạo bảng phương pháp tốt nhất theo từng dataset

In [ ]:
best = summary.sort_values(
    ["dataset", "macro_f1_mean", "accuracy_mean"],
    ascending=[True, False, False]
).groupby("dataset").head(1)

best.to_csv("results/best_method_by_dataset.csv", index=False)

display(best)
print("Saved: results/best_method_by_dataset.csv")

## 16. Tạo file ZIP chứa kết quả

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!zip -r results_colab.zip results runs -x "runs/**/*.keras"

print("Saved: results_colab.zip")

## 17. Gợi ý cập nhật GitHub sau khi có kết quả

Nếu muốn đưa notebook đã chỉnh sửa lên GitHub, làm trên máy local:

```powershell
git add notebooks/colab_run.ipynb
git commit -m "Update Colab notebook for full experiments"
git push origin main
```

Không nên push toàn bộ `runs/` và `results/` nếu repo đang `.gitignore` các thư mục này.